# Brain Tumor Capstone: Full Demo

Research demonstration that connects the repository's trained BRISC 2D models, the supplied SwinUNETR checkpoint contract, anatomy/risk prerequisites, and local RAG implementation. Outputs are experimental and are not clinical advice. Run top to bottom in Google Colab. Four aligned 3D modalities are required; one image is accepted only for the separately trained 2D path.

**Stages report their own status.** A failed or skipped prerequisite does not create a substitute result. No patient scans are included in this notebook or source snapshot.

## A. Runtime, hardware, source and dependencies

In [ ]:
import os, sys, json, shutil, subprocess, platform, resource, time, zipfile
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
print('Colab:', IN_COLAB, '| Python:', platform.python_version())
try:
 import psutil; print('System RAM GiB:', round(psutil.virtual_memory().total/2**30, 2))
except Exception: print('System RAM: psutil unavailable')
if shutil.which('nvidia-smi'):
 print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
else: print('GPU: unavailable; CPU inference will be used where supported')


In [ ]:
# Colab: clone your repository, or upload the provided clean source snapshot ZIP.
REPO_URL = 'https://github.com/REPLACE_WITH_YOUR_ACCOUNT/Capstone_MRI_Project.git'
PROJECT = Path('/content/Capstone_MRI_Project' if IN_COLAB else Path.cwd()).resolve()
if IN_COLAB and not (PROJECT/'README.md').is_file():
 from google.colab import files as colab_files
 if 'REPLACE_WITH_YOUR_ACCOUNT' not in REPO_URL:
  subprocess.run(['git','clone',REPO_URL,str(PROJECT)],check=True)
 else:
  print('Upload Capstone_Source_Snapshot.zip from the project deliverables.')
  uploaded=colab_files.upload(); archives=[name for name in uploaded if name.lower().endswith('.zip')]
  if len(archives)!=1: raise ValueError('Upload exactly one source snapshot ZIP.')
  with zipfile.ZipFile(archives[0]) as archive: archive.extractall(PROJECT)
os.chdir(PROJECT)
print('Project:', PROJECT)


In [ ]:
import src
for _stage in ['1_data_and_preprocessing','2_3d_segmentation','3_2d_segmentation','4_2d_classification']:
 _package_path=str(PROJECT/_stage/'src')
 if _package_path not in src.__path__: src.__path__.append(_package_path)


In [ ]:
# Keep Colab's preinstalled torch intact; install project runtime requirements around it.
%pip install -q 'nibabel>=5,<6' 'monai==1.4.0' 'scipy>=1.10' 'matplotlib>=3.7' 'Pillow>=10' 'PyYAML>=6' 'reportlab>=4' 'psutil>=5.9' 'huggingface_hub>=0.24'
%pip install -q 'rank-bm25>=0.2' 'python-dotenv>=1' 'pydantic>=2.7' 'httpx>=0.27' 'PyMuPDF>=1.24'


In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt, nibabel as nib
from IPython.display import display, Markdown, Image
if IN_COLAB:
 from google.colab import files as colab_files
print('torch',torch.__version__,'| MONAI',__import__('monai').__version__,'| device', 'cuda' if torch.cuda.is_available() else 'cpu')


## Persistent artifacts (optional Google Drive)

In [ ]:
USE_DRIVE = False
if IN_COLAB and USE_DRIVE:
 from google.colab import drive
 drive.mount('/content/drive')
 ARTIFACTS = Path('/content/drive/MyDrive/Capstone_Demo_Artifacts')
else: ARTIFACTS = Path('/content/Capstone_Demo_Artifacts' if IN_COLAB else './Capstone_Demo_Artifacts').resolve()
ARTIFACTS.mkdir(parents=True,exist_ok=True)


## B. Upload and validate one study

In [ ]:
# Upload individual NIfTI files or one ZIP. ZIP members are selected by explicit T1/T1ce/T2/FLAIR token, never upload order.
UPLOAD_DIR = ARTIFACTS/'input'; UPLOAD_DIR.mkdir(exist_ok=True)
if IN_COLAB:
 uploaded = colab_files.upload()
 for name,data in uploaded.items(): (UPLOAD_DIR/name).write_bytes(data)
else: print('Place .nii/.nii.gz files in',UPLOAD_DIR)
for z in UPLOAD_DIR.glob('*.zip'):
 with zipfile.ZipFile(z) as archive: archive.extractall(UPLOAD_DIR/'unzipped')


In [ ]:
# Edit paths explicitly. Filenames can be mapped here regardless of their uploaded order.
STUDY_ID = 'demo-study'
INPUT_PREPROCESSING_CONFIRMED = False  # Set True only if these are one skull-stripped, co-registered study.
MODALITY_PATHS = {'T1':'', 'T1ce':'', 'T2':'', 'FLAIR':''}
# Example: MODALITY_PATHS['T1'] = str(UPLOAD_DIR/'case_T1.nii.gz')


In [ ]:
sys.path.insert(0,str(PROJECT/'notebooks'))
from colab_demo_helpers import validate_nifti_study, sha256, write_json, make_artifact_zip
module_status = {k:{'status':'Skipped because a prerequisite is missing'} for k in ['input','seg2d','class2d','seg3d','gradcam','anatomy_registration','risk','corridor','rag','report']}
input_meta = None
try:
 input_meta = validate_nifti_study(MODALITY_PATHS)
 module_status['input']={'status':'Completed',**input_meta}
 display(Markdown('**Input validation: Completed.** All four explicit modality files have matching dimensions and affine.'))
except Exception as e:
 module_status['input']={'status':'Failed','error':str(e)}
 display(Markdown('**Input validation: Failed.** '+str(e)))


## Checkpoints and configuration provenance

In [ ]:
CHECKPOINTS = {
 'seg2d': PROJECT/'3_2d_segmentation/outputs/segmentation2d/02_attention_boundary_unet_import_20261019_1209/best.pt',
 'class2d': PROJECT/'4_2d_classification/outputs/classification/20261019_104128_seed2026/best.pt',
 'seg3d': PROJECT/'2_3d_segmentation/outputs/segmentation3d/SwinUNETR/best.pth'}
for k,p in CHECKPOINTS.items(): print(k, 'present' if p.is_file() else 'MISSING', p)


In [ ]:
# 2D checkpoints are local-only and omitted from the source ZIP; these upload names are explicit.
if IN_COLAB and (not CHECKPOINTS['seg2d'].is_file() or not CHECKPOINTS['class2d'].is_file()):
 print('Upload seg2d_best.pt and/or class2d_best.pt if available. SwinUNETR is fetched from Hugging Face if absent.')
 uploaded=colab_files.upload()
 names={'seg2d_best.pt':'seg2d','class2d_best.pt':'class2d'}
 for name,data in uploaded.items():
  if name in names:
   target=ARTIFACTS/name; target.write_bytes(data); CHECKPOINTS[names[name]]=target
if not CHECKPOINTS['seg3d'].is_file():
 from huggingface_hub import hf_hub_download
 try:
  CHECKPOINTS['seg3d']=Path(hf_hub_download(repo_id='janiharman/swinunetr-pretrained-brats2025',filename='model_best_fold_0.pth',revision='main',cache_dir='/content/hf_cache' if IN_COLAB else None))
  print('Downloaded cached 754 MB model-card artifact; Hub verifies transfer.')
 except Exception as e: print('SwinUNETR download unavailable:',repr(e))


## C. 2D segmentation (BRISC attention U-Net)

In [ ]:
# The trained BRISC checkpoint expects a 2D grayscale slice, resize 256x256, [-1,1], sigmoid >=0.5.
if not CHECKPOINTS['seg2d'].is_file(): module_status['seg2d']={'status':'Skipped because a prerequisite is missing','reason':'best.pt missing'}
elif not MODALITY_PATHS.get('T1'): module_status['seg2d']={'status':'Skipped because a prerequisite is missing','reason':'select a 2D NIfTI slice in MODALITY_PATHS[T1]'}
else:
 try:
  import torch.nn.functional as F
  sys.path.insert(0,str(PROJECT/'3_2d_segmentation')); sys.path.insert(0,str(PROJECT/'1_data_and_preprocessing'))
  from src.segmentation2d.model_attention import BRISCAttentionUNet
  ck=torch.load(CHECKPOINTS['seg2d'],map_location='cpu',weights_only=True)
  required={'model_state','model_config','image_size','mask_threshold'}
  if not required.issubset(ck): raise ValueError(f'Checkpoint missing keys: {sorted(required-set(ck))}')
  model=BRISCAttentionUNet(base_channels=int(ck['model_config'].get('base_channels',16)))
  model.load_state_dict(ck['model_state'],strict=True); model.eval().to('cuda' if torch.cuda.is_available() else 'cpu')
  vol=nib.load(MODALITY_PATHS['T1']); arr=vol.get_fdata(dtype=np.float32); sl=arr[:,:,arr.shape[2]//2]
  lo,hi=np.percentile(sl[np.isfinite(sl)],(1,99)); disp=np.clip((sl-lo)/max(hi-lo,1e-8),0,1)
  x=torch.from_numpy(disp.astype('float32'))[None,None]; x=F.interpolate(x,size=(int(ck['image_size']),)*2,mode='bilinear',align_corners=False); x=(x-.5)/.5
  with torch.inference_mode(): prob=torch.sigmoid(model(x.to(next(model.parameters()).device)))[0,0].cpu().numpy()
  pred=prob>=.5; fig,ax=plt.subplots(1,3,figsize=(12,4)); ax[0].imshow(disp,cmap='gray'); ax[0].set_title('T1 mid-slice'); ax[1].imshow(prob,cmap='magma'); ax[1].set_title('Probability'); ax[2].imshow(x[0,0].numpy()*.5+.5,cmap='gray'); ax[2].imshow(pred,alpha=.4,cmap='Reds'); ax[2].set_title('Predicted foreground'); [a.axis('off') for a in ax]; plt.savefig(ARTIFACTS/'seg2d_overlay.png',dpi=160,bbox_inches='tight'); plt.show()
  module_status['seg2d']={'status':'Completed','model':'BRISCAttentionUNet','sha256':sha256(CHECKPOINTS['seg2d']),'image_size':int(ck['image_size']),'mask_threshold':.5,'foreground_pixels':int(pred.sum()),'input_slice':'T1 axial midpoint'}
 except Exception as e: module_status['seg2d']={'status':'Failed','error':repr(e)}; print('2D segmentation failed:',repr(e))


## D. 2D classification (BRISC four-class model)

In [ ]:
# Run the actual BRISC classifier on an explicitly selected NIfTI slice as an out-of-domain research demonstration.
if not CHECKPOINTS['class2d'].is_file(): module_status['class2d']={'status':'Skipped because a prerequisite is missing','reason':'best.pt missing'}
elif not MODALITY_PATHS.get('T1'): module_status['class2d']={'status':'Skipped because a prerequisite is missing','reason':'select T1 NIfTI'}
else:
 try:
  from PIL import Image
  sys.path.insert(0,str(PROJECT/'4_2d_classification')); sys.path.insert(0,str(PROJECT/'1_data_and_preprocessing'))
  from src.classification.engine import _make_model
  from src.data.brisc import BRISCClassificationDataset, ClassificationRecord, CLASS_NAMES
  ck=torch.load(CHECKPOINTS['class2d'],map_location='cpu',weights_only=True)
  if ck.get('class_names')!=list(CLASS_NAMES): raise ValueError('Checkpoint label order mismatch')
  cfg={'model':ck['model_config'],'image_size':int(ck['image_size']),'resize_mode':ck.get('preprocessing',{}).get('resize_mode','stretch'),'augmentation':ck.get('preprocessing',{}).get('augmentation',{})}
  model=_make_model(cfg); model.load_state_dict(ck['model_state'],strict=True); device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); model.to(device).eval()
  arr=nib.load(MODALITY_PATHS['T1']).get_fdata(dtype=np.float32); sl=arr[:,:,arr.shape[2]//2]; lo,hi=np.percentile(sl[np.isfinite(sl)],(1,99)); gray=np.uint8(np.clip((sl-lo)/max(hi-lo,1e-8),0,1)*255)
  image_path=ARTIFACTS/'classifier_input.png'; Image.fromarray(gray).save(image_path)
  tensor,_=BRISCClassificationDataset([ClassificationRecord(image_path,0)],cfg['image_size'],resize_mode=cfg['resize_mode'],augment=False,augmentation=cfg['augmentation'])[0]
  temp=float(ck.get('temperature',1.0))
  with torch.inference_mode(): prob=torch.softmax(model(tensor[None].to(device))/temp,dim=1)[0].cpu().numpy()
  result={name:float(prob[i]) for i,name in enumerate(CLASS_NAMES)}; print('Predicted BRISC class:',CLASS_NAMES[int(prob.argmax())]); print('Scores (out-of-domain for a patient NIfTI slice):',result)
  module_status['class2d']={'status':'Completed','model':'BRISC custom CNN','checkpoint_sha256':sha256(CHECKPOINTS['class2d']),'class_order':list(CLASS_NAMES),'temperature':temp,'scores':result,'input':'T1 axial midpoint resampled to BRISC model preprocessing','interpretation':'Out-of-domain demonstration only; scores are not patient-level diagnostic probabilities.','clinical_validation':'Not clinically validated'}
 except Exception as e: module_status['class2d']={'status':'Failed','error':repr(e)}; print('Classification failed:',repr(e))


## E. 3D SwinUNETR segmentation (model-card contract)

In [ ]:
# Model-card contract supplied with this task: input [T1,T1ce,T2,FLAIR], outputs [ET,TC,WT], feature_size=48, nonzero per-channel z-score, sigmoid >=0.5, ROI=128^3, overlap=.7.
# The locally supplied checkpoint records only epoch/state_dict/best_acc. Strict architecture load below is the compatibility gate.


In [ ]:
if input_meta is None or not INPUT_PREPROCESSING_CONFIRMED:
 module_status['seg3d']={'status':'Skipped because a prerequisite is missing','reason':'four aligned modalities not validated, or skull-stripped/co-registered same-study preprocessing was not confirmed'}
elif not CHECKPOINTS['seg3d'].is_file(): module_status['seg3d']={'status':'Skipped because a prerequisite is missing','reason':'checkpoint absent'}
else:
 try:
  from monai.networks.nets import SwinUNETR
  from monai.inferers import sliding_window_inference
  import codecs, collections
  _safe=[(np.core.multiarray.scalar,'numpy._core.multiarray.scalar'),np.dtype,np.ndarray,type(np.dtype(np.float32)),collections.OrderedDict,codecs.encode]
  with torch.serialization.safe_globals(_safe): ck=torch.load(CHECKPOINTS['seg3d'],map_location='cpu',weights_only=True)
  if not isinstance(ck,dict) or 'state_dict' not in ck: raise ValueError('Expected state_dict checkpoint')
  class MONAI_Swin_UNet_Generator(torch.nn.Module):
   def __init__(self):
    super().__init__(); self.net=SwinUNETR(img_size=(128,128,128),in_channels=4,out_channels=3,feature_size=48,use_checkpoint=True)
   def forward(self,value): return self.net(value)
  model=MONAI_Swin_UNet_Generator()
  state=ck['state_dict']
  if state and next(iter(state)).startswith('module.'): state={k.removeprefix('module.'):v for k,v in state.items()}
  model.load_state_dict(state,strict=True); device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); model.to(device).eval()
  arrays=[]
  for name in ['T1','T1ce','T2','FLAIR']:
   a=nib.load(MODALITY_PATHS[name]).get_fdata(dtype=np.float32); nz=a!=0
   if not nz.any(): raise ValueError(f'{name}: all-zero modality')
   mu,sd=float(a[nz].mean()),float(a[nz].std()); arrays.append(np.where(nz,(a-mu)/max(sd,1e-8),0).astype('float32'))
  vol=np.stack(arrays); x=torch.from_numpy(vol)[None]; roi=(128,128,128)
  # Reflection/padding is explicit for inputs smaller than the checkpoint ROI.
  pads=[]
  for n in reversed(x.shape[-3:]): pads.extend([0,max(0,roi[len(pads)//2]-n)])
  if any(pads): x=F.pad(x,pads)
  start=time.time()
  with torch.inference_mode(): logits=sliding_window_inference(x.to(device),roi,1,model,overlap=.7)
  probs=torch.sigmoid(logits)[0].cpu().numpy(); pred=probs>=.5; elapsed=time.time()-start
  pred=pred[:,:vol.shape[1],:vol.shape[2],:vol.shape[3]]
  aff=np.asarray(input_meta['affine']); spacing=np.linalg.norm(aff[:3,:3],axis=0); regions=['ET','TC','WT']; voxel_mm3=float(np.prod(spacing)); vols={}
  for i,name in enumerate(regions):
   vols[name]={'voxels':int(pred[i].sum()),'volume_mm3':float(pred[i].sum()*voxel_mm3)}
   nib.save(nib.Nifti1Image(pred[i].astype('uint8'),aff),str(ARTIFACTS/f'{name}.nii.gz'))
  for i,name in enumerate(regions): nib.save(nib.Nifti1Image(probs[i].astype('float32'),aff),str(ARTIFACTS/f'{name}_probability.nii.gz'))
  z=vol.shape[-1]//2; fig,ax=plt.subplots(1,4,figsize=(14,4)); ax[0].imshow(vol[0,:,:,z],cmap='gray'); ax[0].set_title('T1');
  for j,n in enumerate(regions,1): ax[j].imshow(vol[0,:,:,z],cmap='gray'); ax[j].imshow(pred[j-1,:,:,z],alpha=.45,cmap='autumn'); ax[j].set_title(n)
  [a.axis('off') for a in ax]; plt.savefig(ARTIFACTS/'seg3d_regions.png',dpi=160,bbox_inches='tight'); plt.show()
  module_status['seg3d']={'status':'Completed','model':'MONAI SwinUNETR','contract_source':'user-supplied Hugging Face model card','checkpoint_sha256':sha256(CHECKPOINTS['seg3d']),'output_order':regions,'threshold':.5,'inference_seconds':elapsed,'device':str(device),'regions':vols,'clinical_validation':'Not clinically validated'}
 except Exception as e: module_status['seg3d']={'status':'Failed','error':repr(e)}; print('3D segmentation failed (dependent stages remain gated):',repr(e))


## F. Grad-CAM

In [ ]:
# Classification Grad-CAM uses the project's supported BRISCClassifier.features[14] target and predicted-class logit.
if module_status.get('class2d',{}).get('status') == 'Completed':
 try:
  sys.path.insert(0,str(PROJECT))
  from integrated_pipeline.modules.gradcam import classifier_gradcam
  cam_input=nib.load(MODALITY_PATHS['T1']).get_fdata(dtype=np.float32)
  cam_slice=cam_input[:,:,cam_input.shape[2]//2]
  target_class=max(module_status['class2d']['scores'],key=module_status['class2d']['scores'].get)
  cam, target_layer=classifier_gradcam(CHECKPOINTS['class2d'],cam_slice,target_class)
  plt.imsave(ARTIFACTS/'classification_gradcam.png',cam)
  display(Image(filename=str(ARTIFACTS/'classification_gradcam.png')))
  module_status['gradcam']={'status':'Completed','model':'BRISCClassifier','target_class':target_class,'target_layer':target_layer,'scope':'single axial slice','checkpoint_sha256':sha256(CHECKPOINTS['class2d']),'note':'Explains the selected model logit; not a tumor segmentation mask.'}
 except Exception as e: module_status['gradcam']={'status':'Failed','error':repr(e)}; print('Grad-CAM failed:',repr(e))
else: module_status['gradcam']={'status':'Skipped because a prerequisite is missing','reason':'compatible BRISC classifier prediction/checkpoint unavailable'}


## G–H. SynthSeg, registration, V8 risk and corridor planning

In [ ]:
# Actual SynthSeg wrapper invokes Docker FreeSurfer 8.2.0 and requires a separately supplied FreeSurfer license.
module_status['anatomy_registration']={'status':'Skipped because a prerequisite is missing','reason':'No licensed SynthSeg runtime is bundled; wrapper uses Docker and FS_LICENSE. The project has no general registration transform implementation to align arbitrary uploaded anatomical labels.'}
module_status['risk']={'status':'Skipped because a prerequisite is missing','reason':'V8 requires aligned patient MRI, tumor mask, SynthSeg labels and strict affine QC. It has no configured V8 risk template for this arbitrary upload. Atlas priors are not patient-specific anatomy.'}
module_status['corridor']={'status':'Skipped because a prerequisite is missing','reason':'V8 corridor code exists, but needs successful risk/anatomy prerequisites; actual input data are absent.'}
print(json.dumps({k:module_status[k] for k in ['anatomy_registration','risk','corridor']},indent=2))


## I. Retrieval-augmented evidence synthesis

In [ ]:
RAG_ROOT=PROJECT/'6_rag_treatment_planning'; KB=RAG_ROOT/'knowledge_base/chunks.json'
question=input('Evidence question (blank to skip): ').strip()
if not question: module_status['rag']={'status':'Skipped because a prerequisite is missing','reason':'No question supplied'}
elif not KB.is_file(): module_status['rag']={'status':'Skipped because a prerequisite is missing','reason':'knowledge_base/chunks.json absent; source docs say corpus must be downloaded and built'}
else:
 try:
  sys.path.insert(0,str(RAG_ROOT)); os.chdir(RAG_ROOT)
  from src.rag_pipeline import RAGPipeline
  rag=RAGPipeline(); result=rag.query(question,case_id=STUDY_ID,patient_context={'case_id':STUDY_ID,'missing_modalities':[]})
  display(Markdown('### Retrieved evidence and response'+chr(10)+json.dumps(result,indent=2,default=str)))
  (ARTIFACTS/'rag_result.json').write_text(json.dumps(result,indent=2,default=str))
  module_status['rag']={'status':'Completed','warning':result.get('warnings',[]),'retrieved_count':len(result.get('retrieved_evidence',[])),'citation_validation':result.get('verification',{})}
  os.chdir(PROJECT)
 except Exception as e: module_status['rag']={'status':'Failed','error':repr(e)}; os.chdir(PROJECT); print('RAG failed:',repr(e))


## J. Consolidated PDF, JSON, ZIP and downloads

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as PDFImage, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet


In [ ]:
# Include actual generated outputs only. Skipped stages are described without fabricated visuals or metrics.
report_path=ARTIFACTS/'Capstone_Report.pdf'; styles=getSampleStyleSheet(); story=[Paragraph('Brain Tumor Capstone Research Demo',styles['Title']),Paragraph(f'Study ID: {STUDY_ID} | Generated: {time.ctime()}',styles['Normal']),Spacer(1,12),Paragraph('Experimental research output. Not clinically validated; not for treatment decisions.',styles['Heading2'])]
for name,info in module_status.items():
 story += [Paragraph(name.replace('_',' ').title()+': '+str(info.get('status','Skipped')),styles['Heading2']),Paragraph(__import__('xml.sax.saxutils',fromlist=['escape']).escape(str(info)),styles['BodyText']),Spacer(1,6)]
for p in [ARTIFACTS/'seg2d_overlay.png',ARTIFACTS/'seg3d_regions.png',ARTIFACTS/'classification_gradcam.png']:
 if p.exists():
  image=PDFImage(str(p)); image._restrictSize(420,300); story.append(image)
if input_meta: story.append(Paragraph('Input metadata',styles['Heading2'])); story.append(Paragraph(json.dumps(input_meta,default=str),styles['Code']))
SimpleDocTemplate(str(report_path),pagesize=letter).build(story)
write_json(ARTIFACTS/'module_status.json',{'study_id':STUDY_ID,'modules':module_status,'software':{'python':platform.python_version(),'torch':torch.__version__,'numpy':np.__version__}})
zip_path=make_artifact_zip(ARTIFACTS,ARTIFACTS/'Capstone_Artifacts.zip')
print('PDF bytes:',report_path.stat().st_size,'ZIP bytes:',zip_path.stat().st_size)


In [ ]:
if IN_COLAB:
 colab_files.download(str(report_path)); colab_files.download(str(ARTIFACTS/'module_status.json')); colab_files.download(str(zip_path))
else: print('Artifacts:',ARTIFACTS)


## Execution and validation status

This notebook has not been run on a Colab GPU. The checked-in tests and notebook structural checks are reported in `COLAB_IMPLEMENTATION_AUDIT.md`. A genuine patient study is not included; clinical performance validation requires a separately supplied, authorized study and expert review.